In [1]:
"""
plot_day("2025-03-14") -- visual validation of the utility-DP segmentation.

Standalone: reads the source parquet and one (labels, segments) pair off disk,
no notebook state. HA candles + jma, trades drawn entry-fill -> exit-fill,
flat runs shaded. Zoom/pan via plotly.
"""
import os
print(os.getcwd())

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

SOURCE_PATH   = "data/mnq-ha-2sec-labeller.pqt"
SEGMENTS_PATH = "data/labels_dp_2sec_v2/segments_colour_k0.pqt"

# column mapping in the source parquet
COL_ORD     = "timestamp"
COL_DATE    = "date"
COL_TS      = "timestamp"
COL_HAOPEN  = "Open"
COL_HACLOSE = "Close"
COL_HAHIGH  = "High"
COL_HALOW   = "Low"
COL_JMA     = "jma"

/home/vm/pt/hgt-rl/labeller


In [2]:
_src = pd.read_parquet(SOURCE_PATH, columns=[COL_DATE, COL_TS, COL_HAOPEN, COL_HACLOSE, COL_HAHIGH, COL_HALOW, COL_JMA])
_src = _src.rename(columns={COL_DATE: "date", COL_TS: "ts", COL_HAOPEN: "o", COL_HACLOSE: "c", COL_HAHIGH: "h", COL_HALOW: "l", COL_JMA: "jma"})
print(_src.info())
_src = _src.sort_values(["date", "ts"], kind="stable").reset_index(drop=True)
_seg = pd.read_parquet(SEGMENTS_PATH)

<class 'pandas.DataFrame'>
RangeIndex: 14256735 entries, 0 to 14256734
Data columns (total 7 columns):
 #   Column  Dtype         
---  ------  -----         
 0   date    datetime64[us]
 1   ts      datetime64[us]
 2   o       float64       
 3   c       float64       
 4   h       float64       
 5   l       float64       
 6   jma     float64       
dtypes: datetime64[us](2), float64(5)
memory usage: 761.4 MB
None


In [3]:
def plot_day(date, height=750):
    day = _src[_src["date"] == pd.Timestamp(date).date()]
    if day.empty:
        day = _src[_src["date"] == date]
    if day.empty:
        avail = sorted(_src["date"].unique())
        raise ValueError(f"{date} not in source. range {avail[0]}..{avail[-1]}")
    r0 = day.index[0]
    ts = pd.to_datetime(day["ts"])

    fig = make_subplots(rows=1, cols=1)
    fig.add_trace(go.Candlestick(
        x=ts, open=day["o"], high=day["h"], low=day["l"], close=day["c"],
        name="HA", increasing_line_color="#26a69a", decreasing_line_color="#ef5350",
        line=dict(width=1), opacity=0.9))
    fig.add_trace(go.Scatter(x=ts, y=day["jma"], name="jma",
                             line=dict(color="#0000ff", width=1.6)))

    seg = _seg[_seg["start_row"].between(r0, day.index[-1])]
    trades = seg[seg["label"] != 0]
    flats  = seg[seg["label"] == 0]

    pad = (day["h"].values - day["l"].values).mean() * 0.6
    ts_by_row = pd.Series(ts.values, index=day.index)

    for _, sh in flats.iterrows():
        fig.add_vrect(x0=ts_by_row[sh["start_row"]], x1=ts_by_row[sh["end_row"]],
                      fillcolor="rgba(255,165,0,0.10)", line_width=0, layer="below")

    lg = trades[trades["label"] == 1]
    sh = trades[trades["label"] == -1]
    if len(lg):
        efr = lg["entry_fill_row"].astype(int).values
        xfr = lg["exit_fill_row"].astype(int).values
        fig.add_trace(go.Scatter(
            x=ts_by_row[efr].values, y=day["l"].reindex(efr).values - pad,
            mode="markers", name="long entry",
            marker=dict(symbol="triangle-up", color="#00b300", size=11)))
        fig.add_trace(go.Scatter(
            x=ts_by_row[xfr].values, y=day["c"].reindex(xfr).values,
            mode="markers", name="long exit",
            marker=dict(symbol="x", color="#000000", size=7)))
    if len(sh):
        efr = sh["entry_fill_row"].astype(int).values
        xfr = sh["exit_fill_row"].astype(int).values
        fig.add_trace(go.Scatter(
            x=ts_by_row[efr].values, y=day["h"].reindex(efr).values + pad,
            mode="markers", name="short entry",
            marker=dict(symbol="triangle-down", color="#e60000", size=11)))
        fig.add_trace(go.Scatter(
            x=ts_by_row[xfr].values, y=day["c"].reindex(xfr).values,
            mode="markers", name="short exit",
            marker=dict(symbol="x", color="#000000", size=7)))

    n_long = int((trades["label"] == 1).sum())
    n_short = int((trades["label"] == -1).sum())
    util = float(trades["utility"].sum())
    pct_in = float((~_row_is_flat(seg, day.index)).mean())
    fig.update_layout(
        title=f"{pd.Timestamp(date).date()}   longs {n_long}  shorts {n_short}  "
              f"util {util:.1f}  in-position {pct_in:.0%}",
        height=height, xaxis_rangeslider_visible=False,
        legend=dict(orientation="h", y=1.02, x=0),
        margin=dict(l=40, r=20, t=60, b=30))
    fig.show()
    return seg

def _row_is_flat(seg, rows):
    flat = np.zeros(len(rows), dtype=bool)
    base = rows[0]
    for _, s in seg[seg["label"] == 0].iterrows():
        a = int(s["start_row"]) - base
        b = int(s["end_row"]) - base
        flat[max(a, 0):b + 1] = True
    return flat

In [ ]:
TEST_DAY = "2025-11-14"
plot_day(TEST_DAY)